# 🧱 Texture & Surface Analysis
### Developer: Wong Kai Bin

This notebook extracts texture features using **Gray-Level Co-occurrence Matrix (GLCM)** and **Local Binary Patterns (LBP)** from preprocessed mango images, builds a feature dataset, and evaluates ripeness classification performance.

### Step 1: Imports & Configuration

In [2]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

ROOT = '../cleaned_data'
CLASSES = ['unripe', 'fully_ripe', 'overripe']

ModuleNotFoundError: No module named 'cv2'

### Step 2: Fruit Mask Extraction

In [ ]:
# Mask = non-black pixels from background-removed cleaned images
def get_fruit_mask(img, thresh=30):
    return (img.sum(axis=2) > thresh).astype(np.uint8)

### Step 3: GLCM Feature Extraction

In [ ]:
def extract_glcm_features(gray, mask):
    # Apply mask so background is 0
    masked_gray = cv2.bitwise_and(gray, gray, mask=mask)
    
    # Compute GLCM for 1 pixel distance at 0, 45, 90, and 135 degrees
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm = graycomatrix(masked_gray, distances=[1], angles=angles, levels=256, symmetric=True, normed=True)
    
    # Ignore background level (0) co-occurrences to avoid background skew
    glcm_fruit = glcm.copy()
    glcm_fruit[0, :, :, :] = 0
    glcm_fruit[:, 0, :, :] = 0
    
    sum_glcm = np.sum(glcm_fruit, axis=(0, 1), keepdims=True)
    if np.all(sum_glcm == 0):
        glcm_norm = glcm
    else:
        glcm_norm = glcm_fruit / sum_glcm
        
    contrast = np.mean(graycoprops(glcm_norm, 'contrast'))
    correlation = np.mean(graycoprops(glcm_norm, 'correlation'))
    energy = np.mean(graycoprops(glcm_norm, 'energy'))
    homogeneity = np.mean(graycoprops(glcm_norm, 'homogeneity'))
    
    return {
        'glcm_contrast': float(contrast),
        'glcm_correlation': float(correlation),
        'glcm_energy': float(energy),
        'glcm_homogeneity': float(homogeneity)
    }

### Step 4: LBP Feature Extraction

In [ ]:
def extract_lbp_features(gray, mask, P=8, R=1):
    lbp = local_binary_pattern(gray, P=P, R=R, method='uniform')
    
    # Extract LBP values inside the fruit mask only
    mango_lbp = lbp[mask > 0]
    
    if len(mango_lbp) == 0:
        return {'lbp_mean': 0.0, 'lbp_variance': 0.0, 'lbp_entropy': 0.0}
        
    lbp_mean = float(np.mean(mango_lbp))
    lbp_var = float(np.var(mango_lbp))
    
    # Calculate Shannon entropy from normalized histogram
    n_bins = int(lbp.max() + 1)
    hist, _ = np.histogram(mango_lbp, bins=n_bins, range=(0, n_bins), density=True)
    hist = hist[hist > 0]
    lbp_entropy = float(-np.sum(hist * np.log2(hist)))
    
    return {
        'lbp_mean': lbp_mean,
        'lbp_variance': lbp_var,
        'lbp_entropy': lbp_entropy
    }

### Step 5: Full Texture Feature Extraction Pipeline

In [ ]:
def texture_pipeline(img):
    mask = get_fruit_mask(img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    glcm_feats = extract_glcm_features(gray, mask)
    lbp_feats = extract_lbp_features(gray, mask)
    
    feats = {**glcm_feats, **lbp_feats}
    return feats, mask, gray

### Step 6: Visual Inspection on Sample Images

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
for r, cls in enumerate(CLASSES):
    paths = sorted(glob.glob(f'{ROOT}/train/{cls}/*.jpg'))
    if not paths:
        continue
    img = cv2.imread(paths[0])
    feats, mask, gray = texture_pipeline(img)
    
    lbp_img = local_binary_pattern(gray, P=8, R=1, method='uniform')
    lbp_masked = np.where(mask > 0, lbp_img, 0)
    
    axes[r, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[r, 0].set_title(f'{cls} | Original')
    axes[r, 0].axis('off')
    
    axes[r, 1].imshow(gray, cmap='gray')
    axes[r, 1].set_title('Grayscale')
    axes[r, 1].axis('off')
    
    axes[r, 2].imshow(lbp_masked, cmap='nipy_spectral')
    axes[r, 2].set_title('LBP Map')
    axes[r, 2].axis('off')
    
    # Plot LBP Histogram
    mango_lbp = lbp_img[mask > 0]
    axes[r, 3].hist(mango_lbp, bins=10, density=True, color='teal', alpha=0.7)
    axes[r, 3].set_title(f'LBP Hist (Entropy: {feats["lbp_entropy"]:.2f})')

plt.tight_layout()
plt.show()

### Step 7: Build Feature Dataset (Train & Test)

In [ ]:
def build_feature_dataset(split):
    rows = []
    for cls in CLASSES:
        img_paths = sorted(glob.glob(f'{ROOT}/{split}/{cls}/*.jpg'))
        for path in img_paths:
            try:
                img = cv2.imread(path)
                if img is None:
                    continue
                feats, _, _ = texture_pipeline(img)
                feats['filename'] = os.path.basename(path)
                feats['class'] = cls
                feats['split'] = split
                rows.append(feats)
            except Exception as e:
                print(f"Error processing {path}: {e}")
    return pd.DataFrame(rows)

df_train = build_feature_dataset('train')
df_test = build_feature_dataset('test')

print(f"Train samples: {len(df_train)}, Test samples: {len(df_test)}")
df_train.head()

### Step 8: Save Features to CSV

In [ ]:
df_all = pd.concat([df_train, df_test], ignore_index=True)
output_path = '../output/texture_features.csv'
os.makedirs('../output', exist_ok=True)
df_all.to_csv(output_path, index=False)
print(f"Saved extracted features to {output_path}")

### Step 9: Model Training (Random Forest Classifier)

In [ ]:
feature_cols = ['glcm_contrast', 'glcm_correlation', 'glcm_energy', 'glcm_homogeneity',
                'lbp_mean', 'lbp_variance', 'lbp_entropy']

X_train, y_train = df_train[feature_cols], df_train['class']
X_test, y_test = df_test[feature_cols], df_test['class']

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

### Step 10: Model Evaluation

In [ ]:
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc * 100:.2f}%")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, digits=3))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=CLASSES))

### Step 11: Feature Importance Analysis

In [ ]:
importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', color='darkgreen', ax=ax)
ax.set_title('Texture Feature Importance (Random Forest)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()